In [1]:
import sys

sys.path.append('../../')
from ml_factory.datasets.precompute_tokens import precompute_tokens
from ml_factory.utils import merge_parts_to_dir
from ml_factory import DATA_RAW_DIR, DATA_PROCESSED_DIR
from transformers import AutoTokenizer
import torch
from pathlib import Path
from ml_factory.datasets import PromptBERTDataset
from torch.utils.data import Subset, DataLoader
from ml_factory.datasets.sampler import SplitSampler
import numpy as np
import torch

from torch.utils.data import Subset, DataLoader
from transformers import AutoModelForSequenceClassification
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import numpy as np
from torch.utils.data import Subset, DataLoader

In [ ]:
data_path = DATA_RAW_DIR / 'final_data_cleaner.parquet'
output_path = DATA_PROCESSED_DIR / '32_128_processed_mini'

In [ ]:

dataset = PromptBERTDataset(output_path)
sampler_data = torch.load(
    DATA_PROCESSED_DIR / "splitsampler_train_7_test_15.pt",
    weights_only=False
)

print(type(sampler_data["split"]))
print(sampler_data["split"].keys())
import torch
from torch.utils.data import Subset, DataLoader

# Load previously saved split
sampler_data = torch.load(
    DATA_PROCESSED_DIR / "splitsampler_train_7_test_15.pt",
    weights_only=False
)

# Get the actual IDs
train_ids = sampler_data["split"]["train"]
val_ids = sampler_data["split"]["validation"]
test_ids = sampler_data["split"]["test"]

# Create datasets using the saved IDs
train_dataset = Subset(
    dataset,
    dataset.ids_to_position(train_ids)
)

val_dataset = Subset(
    dataset,
    dataset.ids_to_position(val_ids)
)

test_dataset = Subset(
    dataset,
    dataset.ids_to_position(test_ids)
)
# train_loader = DataLoader(train_dataset, shuffle=True, batch_size=100)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=100)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=100)

n = 5
batch_size = 100
seed = 42

# Original 70% training prompt IDs
train_ids = np.array(sampler_data["split"]["train"])

# Shuffle and divide prompts among 5 BERTs
rng = np.random.default_rng(seed)
rng.shuffle(train_ids)

train_id_subsets = np.array_split(train_ids, n)

# Convert prompt IDs -> all corresponding chunk positions
train_subsets = [
    Subset(dataset, dataset.ids_to_position(ids.tolist()))
    for ids in train_id_subsets
]

# DataLoaders
ensemble_train_loaders = [
    DataLoader(subset, batch_size=batch_size, shuffle=True)
    for subset in train_subsets
]
for i, subset in enumerate(train_subsets):
    print(f"BERT {i+1}: {len(train_id_subsets[i])} prompts -> {len(subset)} chunks")

<class 'dict'>
dict_keys(['train', 'validation', 'test'])
BERT 1: 50723 prompts -> 58473 chunks
BERT 2: 50722 prompts -> 58484 chunks
BERT 3: 50722 prompts -> 58454 chunks
BERT 4: 50722 prompts -> 58707 chunks
BERT 5: 50722 prompts -> 58477 chunks


In [3]:
test_loader

In [4]:
MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"

In [7]:
# ============================================================
# CONFIG
# ============================================================

num_models = 5
num_epochs = 5

learning_rate = 2e-5
weight_decay = 0.01

warmup_ratio = 0.10
max_grad_norm = 1.0


patience = 2

base_seed = 42

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [8]:
from transformers import get_linear_schedule_with_warmup

In [9]:
# ============================================================
# STORAGE
# ============================================================

ensemble_models = []
validation_results = []


# ============================================================
# TRAIN ENSEMBLE
# ============================================================

for model_idx, train_loader in enumerate(ensemble_train_loaders):

    print("\n" + "=" * 70)
    print(f"TRAINING BERT {model_idx + 1}/{num_models}")
    print("=" * 70)


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    model_seed = base_seed + model_idx

    torch.manual_seed(model_seed)
    np.random.seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(model_seed)


    # ========================================================
    # CREATE FRESH BERT-MINI
    # ========================================================

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(device)


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )


    # ========================================================
    # SCHEDULER
    # ========================================================

    num_training_steps = (
        len(train_loader) * num_epochs
    )

    num_warmup_steps = int(
        warmup_ratio * num_training_steps
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )


    print(
        f"Training steps: {num_training_steps}"
    )

    print(
        f"Warmup steps: {num_warmup_steps}"
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    best_val_f1 = -float("inf")
    best_model_state = None
    best_epoch = None

    epochs_without_improvement = 0


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(num_epochs):

        # ====================================================
        # TRAINING
        # ====================================================

        model.train()

        train_loss = 0.0

        progress_bar = tqdm(
            train_loader,
            desc=(
                f"BERT {model_idx + 1} | "
                f"Epoch {epoch + 1}/{num_epochs}"
            )
        )


        for batch in progress_bar:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)


            # ------------------------------------------------
            # CLEAR GRADIENTS
            # ------------------------------------------------

            optimizer.zero_grad()


            # ------------------------------------------------
            # FORWARD PASS
            # ------------------------------------------------

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss


            # ------------------------------------------------
            # BACKWARD PASS
            # ------------------------------------------------

            loss.backward()


            # ------------------------------------------------
            # GRADIENT CLIPPING
            # ------------------------------------------------

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_grad_norm
            )


            # ------------------------------------------------
            # OPTIMIZER
            # ------------------------------------------------

            optimizer.step()


            # ------------------------------------------------
            # LEARNING-RATE SCHEDULER
            # ------------------------------------------------

            scheduler.step()


            # ------------------------------------------------
            # TRACK LOSS
            # ------------------------------------------------

            train_loss += loss.item()


            # ------------------------------------------------
            # PROGRESS BAR
            # ------------------------------------------------

            current_lr = scheduler.get_last_lr()[0]

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{current_lr:.2e}"
            )


        # ====================================================
        # AVERAGE TRAINING LOSS
        # ====================================================

        avg_train_loss = (
            train_loss / len(train_loader)
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_loss = 0.0

        all_labels = []
        all_predictions = []
        all_probabilities = []


        with torch.no_grad():

            for batch in val_loader:

                input_ids = batch["tokenized"].to(device)
                attention_mask = batch["attention_masks"].to(device)
                labels = batch["label"].to(device)


                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )


                val_loss += outputs.loss.item()


                logits = outputs.logits


                probabilities = torch.softmax(
                    logits,
                    dim=1
                )[:, 1]


                predictions = torch.argmax(
                    logits,
                    dim=1
                )


                all_labels.extend(
                    labels.cpu().numpy()
                )

                all_predictions.extend(
                    predictions.cpu().numpy()
                )

                all_probabilities.extend(
                    probabilities.cpu().numpy()
                )


        # ====================================================
        # VALIDATION METRICS
        # ====================================================

        avg_val_loss = (
            val_loss / len(val_loader)
        )

        val_accuracy = accuracy_score(
            all_labels,
            all_predictions
        )

        val_precision = precision_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_recall = recall_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_f1 = f1_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_roc_auc = roc_auc_score(
            all_labels,
            all_probabilities
        )

        val_pr_auc = average_precision_score(
            all_labels,
            all_probabilities
        )


        # ====================================================
        # CURRENT LEARNING RATE
        # ====================================================

        current_lr = scheduler.get_last_lr()[0]


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"F1: {val_f1:.4f} | "
            f"PR-AUC: {val_pr_auc:.4f} | "
            f"ROC-AUC: {val_roc_auc:.4f} | "
            f"LR: {current_lr:.2e}"
        )


        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_f1 > best_val_f1:

            best_val_f1 = val_f1
            best_epoch = epoch + 1

            best_model_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if epochs_without_improvement >= patience:

            print(
                f"Early stopping after "
                f"{epoch + 1} epochs."
            )

            break


    # ========================================================
    # RESTORE BEST CHECKPOINT
    # ========================================================

    model.load_state_dict(best_model_state)

    model.to(device)


    # ========================================================
    # STORE MODEL
    # ========================================================

    ensemble_models.append(model)


    validation_results.append({
        "model": model_idx + 1,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1
    })


    # ========================================================
    # REPORT
    # ========================================================

    print(
        f"\nBERT {model_idx + 1} "
        f"best epoch: {best_epoch}"
    )

    print(
        f"BERT {model_idx + 1} "
        f"best validation F1: {best_val_f1:.4f}"
    )


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("ALL 5 BERT-MINI MODELS HAVE BEEN TRAINED")
print("=" * 70)


for result in validation_results:

    print(
        f"BERT {result['model']}: "
        f"Best Epoch = {result['best_epoch']} | "
        f"Best Val F1 = {result['best_val_f1']:.4f}"
    )


TRAINING BERT 1/5


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Training steps: 2925
Warmup steps: 292


BERT 1 | Epoch 1/5:   0%|          | 0/585 [00:00<?, ?it/s]

...

Epoch 3/5 | Train Loss: 0.1035 | Val Loss: 0.1002 | F1: 0.9647 | PR-AUC: 0.9956 | ROC-AUC: 0.9945 | LR: 8.89e-06


BERT 5 | Epoch 4/5:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 0.0864 | Val Loss: 0.0977 | F1: 0.9660 | PR-AUC: 0.9959 | ROC-AUC: 0.9949 | LR: 4.44e-06


BERT 5 | Epoch 5/5:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 0.0776 | Val Loss: 0.0949 | F1: 0.9670 | PR-AUC: 0.9961 | ROC-AUC: 0.9951 | LR: 0.00e+00

BERT 5 best epoch: 5
BERT 5 best validation F1: 0.9670

ALL 5 BERT-MINI MODELS HAVE BEEN TRAINED
BERT 1: Best Epoch = 5 | Best Val F1 = 0.9671
BERT 2: Best Epoch = 5 | Best Val F1 = 0.9671
BERT 3: Best Epoch = 5 | Best Val F1 = 0.9669
BERT 4: Best Epoch = 4 | Best Val F1 = 0.9673
BERT 5: Best Epoch = 5 | Best Val F1 = 0.9670


In [17]:
len(ensemble_models)

5

In [15]:
MODEL_SAVE_DIR = (
    DATA_PROCESSED_DIR / "ensemble_classifier_mini"
)

MODEL_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for i, model in enumerate(ensemble_models):

    model_save_path = (
        MODEL_SAVE_DIR / f"bert_{i + 1}.pt"
    )

    torch.save(
        model.state_dict(),
        model_save_path
    )

    print(
        f"BERT {i + 1} saved to: {model_save_path}"
    )

BERT 1 saved to: c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\ensemble_classifier_mini\bert_1.pt
BERT 2 saved to: c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\ensemble_classifier_mini\bert_2.pt
BERT 3 saved to: c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\ensemble_classifier_mini\bert_3.pt
BERT 4 saved to: c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\ensemble_classifier_mini\bert_4.pt
BERT 5 saved to: c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\ensemble_classifier_mini\bert_5.pt


In [5]:
MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"

MODEL_SAVE_DIR = DATA_PROCESSED_DIR / "ensemble_classifier_mini"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_models = []

for i in range(1, 6):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    state_dict = torch.load(
        MODEL_SAVE_DIR / f"bert_{i}.pt",
        map_location=device
    )

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    loaded_models.append(model)

print(f"Loaded {len(loaded_models)} models.")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Loaded 5 models.


In [21]:
import pandas as pd

In [22]:
validation_results = []

for model_idx, model in enumerate(loaded_models):

    model.eval()

    val_loss = 0.0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            predictions = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    val_precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    val_recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    val_roc_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    val_pr_auc = average_precision_score(
        all_labels,
        all_probabilities
    )

    validation_results.append({
        "model": model_idx + 1,
        "val_loss": avg_val_loss,
        "accuracy": val_accuracy,
        "precision": val_precision,
        "recall": val_recall,
        "f1": val_f1,
        "roc_auc": val_roc_auc,
        "pr_auc": val_pr_auc
    })

validation_df = pd.DataFrame(validation_results)

validation_df

,model,val_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,0.096563,0.964952,0.965403,0.968879,0.967138,0.995091,0.996051
1,2,0.096293,0.964824,0.963509,0.970680,0.967081,0.994994,0.995938
2,3,0.096158,0.964760,0.965753,0.968129,0.966939,0.995101,0.996046
3,4,0.096069,0.965176,0.968385,0.966118,0.967251,0.994998,0.995949
4,5,0.094874,0.964776,0.964863,0.969120,0.966987,0.995116,0.996054


In [23]:
for result in validation_results:
    print(
        f"BERT {result['model']}: "
        f"F1={result['f1']:.4f}, "
        f"Accuracy={result['accuracy']:.4f}, "
        f"ROC-AUC={result['roc_auc']:.4f}, "
        f"PR-AUC={result['pr_auc']:.4f}"
    )

BERT 1: F1=0.9671, Accuracy=0.9650, ROC-AUC=0.9951, PR-AUC=0.9961
BERT 2: F1=0.9671, Accuracy=0.9648, ROC-AUC=0.9950, PR-AUC=0.9959
BERT 3: F1=0.9669, Accuracy=0.9648, ROC-AUC=0.9951, PR-AUC=0.9960
BERT 4: F1=0.9673, Accuracy=0.9652, ROC-AUC=0.9950, PR-AUC=0.9959
BERT 5: F1=0.9670, Accuracy=0.9648, ROC-AUC=0.9951, PR-AUC=0.9961


In [24]:
validation_results = []
validation_predictions = {}

for model_idx, model in enumerate(loaded_models):

    model.eval()

    val_loss = 0.0
    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)[:, 1]

            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)

    # Default 0.5 threshold
    all_predictions = (all_probabilities >= 0.5).astype(int)

    avg_val_loss = val_loss / len(val_loader)

    validation_predictions[model_idx + 1] = {
        "labels": all_labels,
        "probabilities": all_probabilities
    }

    validation_results.append({
        "model": model_idx + 1,
        "val_loss": avg_val_loss,
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(
            all_labels, all_predictions, zero_division=0
        ),
        "recall": recall_score(
            all_labels, all_predictions, zero_division=0
        ),
        "f1": f1_score(
            all_labels, all_predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(
            all_labels, all_probabilities
        ),
        "pr_auc": average_precision_score(
            all_labels, all_probabilities
        )
    })

validation_df = pd.DataFrame(validation_results)

validation_df

,model,val_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
0,1,0.096563,0.964952,0.965403,0.968879,0.967138,0.995091,0.996051
1,2,0.096293,0.964824,0.963509,0.970680,0.967081,0.994994,0.995938
2,3,0.096158,0.964760,0.965753,0.968129,0.966939,0.995101,0.996046
3,4,0.096069,0.965176,0.968385,0.966118,0.967251,0.994998,0.995949
4,5,0.094874,0.964776,0.964863,0.969120,0.966987,0.995116,0.996054


In [25]:
thresholds = np.arange(0.05, 0.51, 0.05)

threshold_results = []

for model_idx in range(1, len(loaded_models) + 1):

    labels = validation_predictions[model_idx]["labels"]
    probabilities = validation_predictions[model_idx]["probabilities"]

    for threshold in thresholds:

        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "model": model_idx,
            "threshold": threshold,
            "precision": precision_score(
                labels, predictions, zero_division=0
            ),
            "recall": recall_score(
                labels, predictions, zero_division=0
            ),
            "f1": f1_score(
                labels, predictions, zero_division=0
            ),
            "accuracy": accuracy_score(
                labels, predictions
            )
        })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,model,threshold,precision,recall,f1,accuracy
0,1,0.05,0.907256,0.991387,0.947457,0.941470
1,1,0.10,0.928022,0.987816,0.956986,0.952732
2,1,0.15,0.938896,0.984965,0.961379,0.957875
3,1,0.20,0.944616,0.982234,0.963058,0.959888
4,1,0.25,0.949429,0.979773,0.964362,0.961454
5,1,0.30,0.953036,0.977432,0.965080,0.962348
6,1,0.35,0.956207,0.975692,0.965851,0.963275
7,1,0.40,0.959027,0.973561,0.966239,0.963786
8,1,0.45,0.962889,0.970980,0.966918,0.964633
9,1,0.50,0.965403,0.968879,0.967138,0.964952


In [8]:
import time
import numpy as np
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

all_model_probabilities = []
test_labels = None

ensemble_start = time.perf_counter()

for model_idx, model in enumerate(loaded_models):

    print(f"\nStarting model {model_idx + 1}/{len(loaded_models)}", flush=True)

    model.eval()

    model_probabilities = []
    model_labels = []

    model_start = time.perf_counter()

    with torch.no_grad():

        for batch_idx, batch in enumerate(test_loader):

            # Print progress every 10 batches
            if batch_idx % 10 == 0:
                print(
                    f"  Model {model_idx + 1}: "
                    f"processing batch {batch_idx}",
                    flush=True
                )

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )[:, 1]

            model_probabilities.extend(
                probabilities.detach().cpu().numpy()
            )

            model_labels.extend(
                labels.detach().cpu().numpy()
            )

    # Synchronize only AFTER this model is finished
    if device.type == "cuda":
        torch.cuda.synchronize()

    model_time = time.perf_counter() - model_start

    all_model_probabilities.append(
        np.array(model_probabilities)
    )

    if test_labels is None:
        test_labels = np.array(model_labels)

    print(
        f"Finished model {model_idx + 1}: "
        f"{len(model_probabilities)} predictions | "
        f"{model_time:.2f} seconds",
        flush=True
    )


if device.type == "cuda":
    torch.cuda.synchronize()

total_inference_time = time.perf_counter() - ensemble_start

print("\n====================================")
print(f"Total inference time: {total_inference_time:.2f} seconds")
print(f"Number of examples: {len(test_labels)}")

latency_per_example = (
    total_inference_time / len(test_labels)
)

print(
    f"Ensemble latency per example: "
    f"{latency_per_example * 1000:.3f} ms"
)

print("====================================")


Starting model 1/5
  Model 1: processing batch 0
  Model 1: processing batch 10
  Model 1: processing batch 20
  Model 1: processing batch 30
  Model 1: processing batch 40
  Model 1: processing batch 50
  Model 1: processing batch 60
  Model 1: processing batch 70
  Model 1: processing batch 80
  Model 1: processing batch 90
  Model 1: processing batch 100
  Model 1: processing batch 110
  Model 1: processing batch 120
  Model 1: processing batch 130
  Model 1: processing batch 140
  Model 1: processing batch 150
  Model 1: processing batch 160
  Model 1: processing batch 170
  Model 1: processing batch 180
  Model 1: processing batch 190
  Model 1: processing batch 200
  Model 1: processing batch 210
  Model 1: processing batch 220
  Model 1: processing batch 230
  Model 1: processing batch 240
  Model 1: processing batch 250
  Model 1: processing batch 260
  Model 1: processing batch 270
  Model 1: processing batch 280
  Model 1: processing batch 290
  Model 1: processing batch 300

In [9]:
num_test_examples = len(test_labels)

latency_per_example = (
    total_inference_time / num_test_examples
)

print(f"Number of test examples: {num_test_examples}")
print(f"Latency per example: {latency_per_example:.6f} seconds")
print(
    f"Latency per example: "
    f"{latency_per_example * 1000:.3f} ms"
)

Number of test examples: 63139
Latency per example: 0.022191 seconds
Latency per example: 22.191 ms
